# Dataset Preprocessing & Vector DB Ingestion
This notebook preprocesses a CSV dataset (Context + Response columns) and stores the embeddings in a Qdrant vector database.

## 1. Imports

In [18]:
import re
import ast
import os

import pandas as pd
from dotenv import load_dotenv
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

## 2. Load Environment Variables

In [19]:
load_dotenv("")

OLD_DS_NAME = os.getenv("RAG_DATASET")
NEW_DS_NAME = os.getenv("GROUPED_RAG_DATASET")

## 3. Preprocessor
Cleans raw retrieval strings by stripping HTML tags (both closed and unclosed/truncated),
which can overflow the context window of LLMs.

> **Dataset schema:** the CSV has exactly 2 columns:
> - `Context` — a question string
> - `Response` — a list of answer strings
>
> The `Preprocessor` is applied to each response string before the data is saved.

In [20]:
class Preprocessor:
    def __init__(self):
        pass

    def process(self, retrieval: str) -> str:
        # remove closed tags and unclosed tags (e.g. <img with truncated base64)
        # since they overflow the context window limit for llms.
        result = re.sub(r"<.*?>", "", retrieval, flags=re.DOTALL)
        result = re.sub(r"<[^>]*$", "", result, flags=re.DOTALL)
        return result.strip()

## 4. Duplicate Helpers

In [21]:
duplicates_counter = 0

def count_duplicates(responses: list[str]):
    global duplicates_counter
    passed = set()
    for i in range(len(responses)):
        if i in passed:
            continue
        for j in range(i + 1, len(responses)):
            if j in passed:
                continue
            if responses[i] == responses[j]:
                duplicates_counter += 1
                passed.add(j)


def remove_duplicates(responses: list[str]):
    passed = set()
    for i in range(len(responses)):
        if i in passed:
            continue
        for j in range(i + 1, len(responses)):
            if j in passed:
                continue
            if responses[i] == responses[j]:
                passed.add(j)

    for i in sorted(list(passed), reverse=True):
        responses.pop(i)

## 5. Load & Clean CSV

In [22]:
df = pd.read_csv(f"../../rag/{OLD_DS_NAME}.csv")


In [23]:
df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [24]:
df[df['Response'] == '']


,Context,Response


In [25]:
df[df["Response"].isna()]

,Context,Response
2434,"From the moment I wake up, I hear what I think...",NaN
3006,I’m trying to make marriage work after a split...,NaN
3223,Every winter I find myself getting sad because...,NaN
3224,Does counseling really do anything that can he...,NaN


In [26]:
df.dropna(inplace=True)


In [27]:
df.shape

(3507, 2)

## 6. Apply Preprocessor to Responses

In [28]:
grouped = df.groupby("Context")["Response"].apply(list).reset_index()


In [29]:
preprocessor = Preprocessor()

grouped["Response"] = grouped["Response"].apply(
    lambda responses: [preprocessor.process(r) for r in responses]
)

## 7. Remove Duplicates

In [30]:
grouped["Response"].apply(count_duplicates)
print("Number of duplicates: ", duplicates_counter)
grouped["Response"].apply(remove_duplicates)
duplicates_counter = 0
grouped["Response"].apply(count_duplicates)
print("Number of duplicates: ", duplicates_counter)

Number of duplicates:  1061
Number of duplicates:  0


## 8. Save Grouped CSV

In [31]:
grouped.to_csv(f"../../rag/{NEW_DS_NAME}.csv")

## 9. Set Up Embeddings & Qdrant Client

In [32]:
DS_NAME = os.getenv("GROUPED_RAG_DATASET")

embeddings = HuggingFaceEmbeddings(
    model_name=os.getenv("EMBEDDING_MODEL"),
    model_kwargs={"device": os.getenv("DEVICE")},
    encode_kwargs={"normalize_embeddings": True},
)

client = QdrantClient(
    url=os.getenv("QDRANT_CLUSTER_ENDPOINT"),
    api_key=os.getenv("QDRANT_API_KEY"),
    timeout=120,
)

collection_name = os.getenv("EMBEDDINGS_COLLECTION_NAME")

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=int(os.getenv("EMBEDDING_SIZE")),
            distance=Distance.COSINE,
        ),
    )

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9795.39it/s]


## 10. Build Documents

In [33]:
df = pd.read_csv(f"../../rag/{DS_NAME}.csv")
df = df.dropna(subset=["Response"])

docs = [
    Document(
        page_content=row["Context"],
        metadata={"Response": ast.literal_eval(row["Response"])},
    )
    for _, row in df.iterrows()
]

## 11. Store in Vector DB

In [34]:
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)


def chunk_data(data, size=256):
    for i in range(0, len(data), size):
        yield data[i : i + size]


total_inserted = 0

for batch_num, batch in enumerate(chunk_data(docs, size=256), start=1):
    vectorstore.add_documents(batch)
    total_inserted += len(batch)
    print(
        f"Batch {batch_num}: inserted {len(batch)} docs "
        f"(total={total_inserted}/{len(docs)})"
    )

print(f"Inserted {total_inserted} documents")

KeyboardInterrupt: 